# 这里介绍真实开发场景怎么使用

和大模型结合起来， 交互比较复杂， 这里有官方推荐的，好的，推荐使用的方式
提供了预定义状态模型langgraph.graph.message.MessagesState， 可以直接使用

开发的时候可以继承这个状态类型，然后拓展自定义字段
这个源代码 只有messages
```
class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]
```

AnyMessage 是 langchain_core.messages.AnyMessage 的一个别名， 可以是 HumanMessage, AIMessage, SystemMessage ToolMessage

add_messages 就是前面说的常用内置reducer函数

add_message完整限定名是 langgraph.graph.message.add_message

In [6]:
from langchain_core.messages import HumanMessage
# 运行一下， langgraph和deepseek结合
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import MessagesState
from typing import TypedDict
from langchain_deepseek import ChatDeepSeek # 用langchanin的方式使用

from dotenv import load_dotenv
load_dotenv(override=True) # 覆盖写入， 如果有参数一样，就覆盖掉

# 自己调用，还是要导入依赖load_dotnet找key了

# 0. 连接LLM模型
model = ChatDeepSeek( # 有好处自己去调用env的key
    model="deepseek-v4-flash",
    extra_body={
        "thinking": {"type": "disabled"}  # 关闭思考模式，反应快一点
    }
)
# thinking需要看文档https://api-docs.deepseek.com/zh-cn/guides/thinking_mode/

# 1. 定义状态
class OverAllState(MessagesState):
    username: str
    output: str


# 2. 定义节点
def node_a(state: OverAllState) -> OverAllState:  # 先用全局类型，结构简单一点
    return {
         "messages": [HumanMessage("你好，我是" + state["username"])] # 人类询问的话
    }

myTemp = None

def llm_node(state: OverAllState) -> OverAllState:
    res = model.invoke(state["messages"]) # 为什么要讲预定义模型Message，因为可以直接给大预言模型发状态 ，取state["messages"]就行， 比较方便
    print(res)
    print("\n\n\n\n\n\n")

    global myTemp
    myTemp = res

    print([res])
    print("\n\n\n\n\n\n")
    return {
        "messages": [res],  # 返回方便，直接更新messages
        "output": res.content  # content是大预言模型回答的话
    }

# 3. 构建图
builder = StateGraph(state_schema=OverAllState)

builder.add_node("node_a",node_a)
builder.add_node("llm_node",llm_node)

builder.add_edge("node_a","llm_node")
builder.add_edge("llm_node",END)
builder.add_edge(START,"node_a")

graph = builder.compile()
result = graph.invoke({"username":"老王"})
print(result)


content='你好老王！👋 很高兴见到你。有什么我可以帮你的吗？无论是聊聊近况、解答疑问，还是帮你处理一些文字工作，我都在这里。尽管开口吧！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 39, 'prompt_tokens': 8, 'total_tokens': 47, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e', 'id': 'ba8b01c4-b54c-455d-a229-f5a3a1461ce8', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07f7d-4c0e-7891-b42d-e784cad65d08-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'output_tokens': 39, 'total_tokens': 47, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}







[AIMessage(content='你好老王！👋 很高兴见到你。有什么我可以帮你的吗？无论是聊聊近况、解答疑问，还是帮你处理一些文字工作，我都在这里。尽管开口吧！', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_t

你的观察非常敏锐！这确实是 Python 的 `__str__` / `__repr__` 机制在起作用，而且涉及**三层不同的调用链**。让我逐一拆解你看到的三种输出：

---

### 1️⃣ `print(res)` → 调用 `res.__str__()`

```
content='你好老王！👋 ...' additional_kwargs={...} response_metadata={...} id='...' ...
```

-   `print()` 优先调用 `__str__()`
-   LangChain 的 `AIMessage` 继承自 Pydantic `BaseModel`，其 `__str__` 被重写为：**只展示字段键值对，不带类名**
-   所以你看不到 `AIMessage(...)` 这个外壳，直接是 `content=... additional_kwargs=...`

### 2️⃣ `print([res])` → 列表的 `__str__()` 内部对元素调用 `res.__repr__()`

```
[AIMessage(content='你好老王！👋 ...', additional_kwargs={...}, ...)]
```

-   `print([res])` → 调用 `list.__str__()`
-   **关键规则**：Python 容器（list/dict/tuple）在格式化自身字符串时，对内部元素一律调用 `repr()` 而非 `str()`
-   `AIMessage.__repr__()` 返回的是 **带类名的完整构造形式**：`AIMessage(content=..., ...)`
-   所以你看到了 `AIMessage(...)` 包裹 + 外层 `[...]`

> 📌 这就是为什么同一个对象，放进列表里打印就"变了样"——不是对象变了，是**容器选择了 `__repr__` 协议**。

### 3️⃣ `print(result)` → dict 的 `__str__()` 对所有值递归调用 `repr()`

```
{'messages': [HumanMessage(...), AIMessage(...)], 'username': '老王', 'output': '...'}
```

-   `result` 是一个 `dict`
-   同理，`dict.__str__()` 对每个 value 调用 `repr()`
-   所以 `messages` 列表里的消息也全部以 `HumanMessage(...)` / `AIMessage(...)` 形式显示
-   而 `'username'` 和 `'output'` 是普通 str，`repr(str)` 会带引号，但因为它们是纯 ASCII/中文，视觉上和内容一致

---

### 🔬 验证实验

你可以用以下代码确认这个机制：

```python
# 验证容器内调用的是 repr 而非 str
class Demo:
    def __str__(self):
        return "STR版本"
    def __repr__(self):
        return "REPR版本"

d = Demo()
print(d)       # STR版本
print([d])     # [REPR版本]  ← 列表用了 repr
print({"k": d}) # {'k': REPR版本} ← 字典也用了 repr
```

---

### 📊 总结对照表

| 表达式 | 实际调用链 | 输出特征 |
| :--- | :--- | :--- |
| `print(res)` | `res.__str__()` | 无类名，纯字段键值对 |
| `print([res])` | `list.__str__()` → `res.__repr__()` | 有 `AIMessage(...)` 类名 + `[...]` |
| `print(result)` | `dict.__str__()` → 各value的 `__repr__()` | 所有 Message 都带类名 |

> 💡 **实用建议**：调试时如果想看完整的对象结构，始终用 `repr(obj)` 或 `print(f"{obj!r}")`；如果只想看内容摘要，用 `print(obj)`。LangChain 故意这样设计，就是让 `print(msg)` 更可读，而 `print([msg])` 保留可重建的完整信息。

{
    'messages': [
        HumanMessage(content='你好，我是老王',
        additional_kwargs={

        },
        response_metadata={

        },
        id='8e419ab1-da56-4ecd-b145-80afbfcb5b54'),
        AIMessage(content='老王你好呀！👋我是DeepSeek，很高兴见到你。有什么我可以帮你的吗？不管是聊天、解答问题、写点东西，还是帮你处理文件，我都很乐意效劳！\n\n你今天想聊点什么，还是有什么具体的事情需要帮忙？尽管说，我在这儿等着呢！😊',
        additional_kwargs={
            'refusal': None
        },
        response_metadata={ # 返回元数据
            'token_usage': {
                'completion_tokens': 65,
                'prompt_tokens': 8,
                'total_tokens': 73,
                'completion_tokens_details': None,
                'prompt_tokens_details': {
                    'audio_tokens': None,
                    'cached_tokens': 0
                },
                'prompt_cache_hit_tokens': 0,
                'prompt_cache_miss_tokens': 8
            },
            'model_provider': 'deepseek',
            'model_name': 'deepseek-v4-flash',
            'system_fingerprint': 'a26a7955944dc5c60445bff77fac9c8e',
            'id': '16265a74-259a-4c3b-9e1f-84f039838396',
            'finish_reason': 'stop',
            'logprobs': None
        },
        id='lc_run--01a07f22-2b48-72c2-ad56-3ae569b48c9a-0',
        tool_calls=[

        ],
        invalid_tool_calls=[

        ],
        usage_metadata={
            'input_tokens': 8,
            'output_tokens': 65,
            'total_tokens': 73,
            'input_token_details': {
                'cache_read': 0
            },
            'output_token_details': {

            }
        })
    ],
    'username': '老王',
    'output': '老王你好呀！👋我是DeepSeek，很高兴见到你。有什么我可以帮你的吗？不管是聊天、解答问题、写点东西，还是帮你处理文件，我都很乐意效劳！\n\n你今天想聊点什么，还是有什么具体的事情需要帮忙？尽管说，我在这儿等着呢！😊'
}


返回的结果，会有一部分是增量更新的内容
1. id 很重要， 需要自己进行维护， id一样，会被覆盖
2. HumanMessage 自己发送的消息
3. AIMessage 大预言模型回答的消息
预定义状态可以方便后面和大预言模型沟通

如果自己去维护状态记录很多字段，不方便，官方就有很多字段